# XGBoost Training — Safe Lag v3

Self-contained pipeline: load từ `splits_safe_lag_v3/`, baseline training, Optuna tuning, retrain, evaluate.

| | v2 (cũ) | v3 (mới) |
|---|---|---|
| Removed | 6 toxic lags | 8 (thêm rolling_mean_28, rolling_std_7) |
| Added | lag_16, rolling_mean_30 | lag_16/21/35, rolling_mean_30/28_safe, rolling_std_28_safe |
| Features | 45 | 47 |

## 1. Load Dataset and Import Libraries

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_here] + list(_here.parents) if (p / 'config.py').exists()),
    _here
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import PROJECT_ROOT, DATA_DIR, RAW_DIR, PROCESSED_DIR, SPLITS_DIR

MODELS_DIR    = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
SPLITS_V3     = PROCESSED_DIR / 'splits_safe_lag_v3'

assert SPLITS_V3.exists(), (
    f'splits_safe_lag_v3 không tồn tại: {SPLITS_V3}\n'
    f'Hãy chạy feature_safe_lag_v3.ipynb trước!'
)
print(f'SPLITS_V3  : {SPLITS_V3}')
print(f'MODELS_DIR : {MODELS_DIR}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_train    = pd.read_csv(SPLITS_V3 / 'train_features.csv')
y_train    = pd.read_csv(SPLITS_V3 / 'train_target.csv')
X_val      = pd.read_csv(SPLITS_V3 / 'val_features.csv')
y_val      = pd.read_csv(SPLITS_V3 / 'val_target.csv')
X_test     = pd.read_csv(SPLITS_V3 / 'test_features.csv')
y_test     = pd.read_csv(SPLITS_V3 / 'test_target.csv')
y_val_orig = pd.read_csv(SPLITS_V3 / 'val_target_original.csv')

assert X_train.shape[1] == 47, f'Expected 47 cols, got {X_train.shape[1]}'

v3_features = ['lag_16', 'lag_21', 'lag_35', 'rolling_mean_30', 'rolling_mean_28_safe', 'rolling_std_28_safe']
removed     = ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']

print('v3 features present:')
for col in v3_features:
    print(f'  {col:<25}: {"YES ✓" if col in X_train.columns else "MISSING ✗"}')
print('Removed features absent:')
for col in removed:
    print(f'  {col:<25}: {"absent ✓" if col not in X_train.columns else "STILL PRESENT ✗"}')

print(f'\nX_train : {X_train.shape}   (expected 47 cols)')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')

## 1b. Encode Categorical Features

In [ ]:
object_cols    = X_train.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}

for col in object_cols:
    le    = LabelEncoder()
    parts = [X_train[col], X_val[col]]
    if col in X_test.columns:
        parts.append(X_test[col])
    combined = pd.concat(parts, ignore_index=True).astype(str)
    le.fit(combined)

    X_train[col] = le.transform(X_train[col].astype(str)).astype(np.int32)
    X_val[col]   = le.transform(X_val[col].astype(str)).astype(np.int32)
    if col in X_test.columns:
        X_test[col] = le.transform(X_test[col].astype(str)).astype(np.int32)

    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} labels -> 0..{len(le.classes_) - 1}')

rest = X_train.select_dtypes(include=['object']).columns.tolist()
if rest:
    raise ValueError(f'Still has object columns: {rest}')

print(f'\nLabelEncoded {len(object_cols)} cols | '
      f'X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}')

## 2. XGBoost Training (Baseline)

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror',
    tree_method='hist',
    early_stopping_rounds=100,
)

model.fit(
    X_train, y_train['sales_log'],
    eval_set=[(X_val, y_val['sales_log'])],
    verbose=100,
)

## 3. Baseline Evaluation (Before Optuna Tuning)

In [ ]:
y_test_orig = pd.DataFrame()
y_test_orig['sales'] = np.expm1(y_test['sales_log'])


def evaluate_metrics(y_true_log, y_pred_log, y_true_orig, label=''):
    y_pred_orig = np.clip(np.expm1(y_pred_log), 0, None)
    y_true_orig = np.clip(y_true_orig, 0, None)

    rmsle    = np.sqrt(mean_squared_log_error(y_true_orig, y_pred_orig))
    rmse     = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    mae      = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    if label:
        print(f'\nXGBoost — {label}:')
        print(f'  RMSLE              : {rmsle:.6f}')
        print(f'  RMSE (sales units) : {rmse:.6f}')
        print(f'  MAE  (sales units) : {mae:.6f}')
        print(f'  RMSE (log scale)   : {rmse_log:.6f}')

    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'RMSE_log': rmse_log}


baseline_val  = evaluate_metrics(
    y_val['sales_log'],
    model.predict(X_val),
    y_val_orig['sales'],
    'Baseline — Val'
)

baseline_test = evaluate_metrics(
    y_test['sales_log'],
    model.predict(X_test),
    y_test_orig['sales'],
    'Baseline — Test'
)

baseline_df = pd.DataFrame([baseline_val, baseline_test],
    index=['XGBoost Val', 'XGBoost Test']
)

print('\nBaseline Table:')
display(baseline_df.round(4))

## 4. Optuna Tuning

In [ ]:
import json
import os
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    params = {
        'n_estimators'        : 1000,
        'learning_rate'       : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth'           : trial.suggest_int('max_depth', 4, 10),
        'min_child_weight'    : trial.suggest_int('min_child_weight', 20, 300),
        'subsample'           : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'    : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'           : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda'          : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'objective'           : 'reg:squarederror',
        'tree_method'         : 'hist',
        'early_stopping_rounds': 50,
        'random_state'        : 42,
        'n_jobs'              : -1,
    }

    trial_model = xgb.XGBRegressor(**params)
    trial_model.fit(
        X_train, y_train['sales_log'],
        eval_set=[(X_val, y_val['sales_log'])],
        verbose=False,
    )

    val_preds = trial_model.predict(X_val)
    rmsle = np.sqrt(mean_squared_log_error(
        np.clip(y_val_orig['sales'], 0, None),
        np.clip(np.expm1(val_preds), 0, None)
    ))
    return rmsle


params_file = MODELS_DIR / 'xgboost_safe_lag_v3_optuna_best_params.json'
if os.path.exists(params_file):
    print(f'Found {params_file.name}. Skipping Optuna tuning...')
    with open(params_file, 'r') as f:
        best_optuna_params = json.load(f)
else:
    print('Starting Optuna tuning...')
    study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=45, show_progress_bar=True)
    best_optuna_params = study.best_params
    print(f'\nBest RMSLE on val : {study.best_value:.6f}')
    os.makedirs(params_file.parent, exist_ok=True)
    with open(params_file, 'w') as f:
        json.dump(best_optuna_params, f)
    print(f'Saved best parameters to {params_file}')

print('\nBest params:')
for k, v in best_optuna_params.items():
    print(f'  {k}: {v}')

## 5. Retrain with the Tuned Parameters

In [ ]:
import joblib

X_train_val     = pd.concat([X_train, X_val], ignore_index=True)
y_train_val_log = pd.concat(
    [y_train['sales_log'], y_val['sales_log']], ignore_index=True
)

best_params = {
    **best_optuna_params,
    'n_estimators': 1000,
    'objective'   : 'reg:squarederror',
    'tree_method' : 'hist',
    'random_state': 42,
    'n_jobs'      : -1,
}

model_file = MODELS_DIR / 'xgboost_safe_lag_v3_model.pkl'
if os.path.exists(model_file):
    print(f'Found {model_file.name}. Loading model...')
    best_model = joblib.load(model_file)
else:
    print('Retraining model with tuned parameters on Train + Val...')
    best_model = xgb.XGBRegressor(**best_params)
    best_model.fit(X_train_val, y_train_val_log, verbose=100)
    os.makedirs(model_file.parent, exist_ok=True)
    joblib.dump(best_model, model_file)
    print(f'Saved model to {model_file}')

tuned_val = evaluate_metrics(
    y_val['sales_log'],
    best_model.predict(X_val),
    y_val_orig['sales'],
    'Tuned — Val'
)

tuned_test = evaluate_metrics(
    y_test['sales_log'],
    best_model.predict(X_test),
    y_test_orig['sales'],
    'Tuned — Test'
)

## 6. Comparison

In [ ]:
metrics_order = ['RMSLE', 'RMSE', 'MAE', 'RMSE_log']

summary = pd.DataFrame({
    'Metric'       : metrics_order,
    'Baseline Val' : [baseline_val[m]  for m in metrics_order],
    'Tuned Val'    : [tuned_val[m]     for m in metrics_order],
    'Val Gap'      : [tuned_val[m]  - baseline_val[m]  for m in metrics_order],
    'Baseline Test': [baseline_test[m] for m in metrics_order],
    'Tuned Test'   : [tuned_test[m]    for m in metrics_order],
    'Test Gap'     : [tuned_test[m] - baseline_test[m] for m in metrics_order],
})

print('\nXGBoost Safe Lag v3 — Before vs After Optuna Tuning')
print('=' * 85)
print(summary.to_string(index=False, float_format='{:.6f}'.format))
print('=' * 85)
print('Note: Negative Gap = improvement after tuning')

## 7. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

top_n  = 20
top_df = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top_df, x='importance', y='feature',
    hue='feature', palette='viridis', legend=False, ax=ax
)
ax.set_title(f'Top {top_n} Feature Importances – XGBoost Safe Lag v3', fontsize=14)
ax.set_xlabel('Importance')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print(f'\nTop {top_n} features:')
print(top_df.to_string(index=False))

## 8. Error Analysis

In [ ]:
y_pred_val      = best_model.predict(X_val)
val_pred_actual = np.maximum(np.expm1(y_pred_val), 0.0)
y_actual        = y_val_orig['sales'].values if isinstance(y_val_orig, pd.DataFrame) else y_val_orig.values

date_val = pd.to_datetime(X_val[['year', 'month', 'day']])

val_df = pd.DataFrame({
    'date'      : date_val.values,
    'store_nbr' : X_val['store_nbr'].values,
    'family'    : X_val['family'].values,
    'actual'    : y_actual,
    'predicted' : val_pred_actual,
})
val_df['residual'] = val_df['actual'] - val_df['predicted']

In [ ]:
daily = pd.DataFrame({
    'date'     : date_val,
    'actual'   : y_val_orig['sales'].values,
    'predicted': val_pred_actual,
}).groupby('date').sum()

plt.figure(figsize=(14, 5))
plt.plot(daily.index, daily['actual'],    label='Actual',    color='#58a6ff')
plt.plot(daily.index, daily['predicted'], label='Predicted', color='#3fb950', linestyle='--')
plt.title('Total Sales — Actual vs Predicted (Val) - XGBoost Safe Lag v3')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
y_pred_test      = best_model.predict(X_test)
test_pred_actual = np.maximum(np.expm1(y_pred_test), 0.0)

date_test = pd.to_datetime(X_test[['year', 'month', 'day']])

test_df = pd.DataFrame({
    'date'      : date_test.values,
    'store_nbr' : X_test['store_nbr'].values,
    'family'    : X_test['family'].values,
    'actual'    : y_test_orig['sales'].values,
    'predicted' : test_pred_actual,
})
test_df['residual'] = test_df['actual'] - test_df['predicted']

daily_test = test_df[['date', 'actual', 'predicted']].groupby('date').sum()

plt.figure(figsize=(14, 5))
plt.plot(daily_test.index, daily_test['actual'],    label='Actual',    color='#58a6ff')
plt.plot(daily_test.index, daily_test['predicted'], label='Predicted', color='#3fb950', linestyle='--')
plt.title('Total Sales — Actual vs Predicted (Test Set) - XGBoost Safe Lag v3')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
try:
    holidays      = pd.read_csv(PROCESSED_DIR / 'holidays_events_cleaned.csv')
    holiday_dates = set(pd.to_datetime(holidays['date']).dt.strftime('%Y-%m-%d'))
    val_df['date_str']   = pd.to_datetime(val_df['date']).dt.strftime('%Y-%m-%d')
    val_df['is_holiday'] = val_df['date_str'].isin(holiday_dates)

    holiday_summary = []
    for label, mask in [('Holiday', val_df['is_holiday']), ('Non-Holiday', ~val_df['is_holiday'])]:
        sub = val_df[mask]
        r   = np.sqrt(mean_squared_log_error(sub['actual'].clip(0), sub['predicted'].clip(0)))
        holiday_summary.append({'label': label, 'rmsle': r, 'n_rows': int(mask.sum())})
        print(f'{label}: RMSLE = {r:.4f} ({int(mask.sum()):,} rows)')

    fig, ax = plt.subplots(figsize=(6, 4))
    df_hol = pd.DataFrame(holiday_summary)
    sns.barplot(data=df_hol, x='label', y='rmsle', palette=['#ffa657', '#58a6ff'], ax=ax)
    ax.set_title('RMSLE: Holiday vs Non-Holiday – XGBoost Safe Lag v3')
    ax.set_xlabel('')
    ax.set_ylabel('RMSLE')
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('holidays_events_cleaned.csv not found — skipping holiday analysis.')

In [ ]:
daily_resid = val_df.groupby('date')['residual'].mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_resid.index, daily_resid.values, color='#9467bd', linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.fill_between(daily_resid.index, daily_resid.values, 0,
                where=daily_resid.values > 0, alpha=0.15, color='#d62728', label='Under-predict')
ax.fill_between(daily_resid.index, daily_resid.values, 0,
                where=daily_resid.values < 0, alpha=0.15, color='#58a6ff', label='Over-predict')
ax.set_title('Mean Daily Residual (actual − predicted) – XGBoost Safe Lag v3')
ax.set_xlabel('Date')
ax.set_ylabel('Residual')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print('\n' + '='*45)
print('ERROR ANALYSIS SUMMARY — XGBoost Safe Lag v3')
print('='*45)
print(f"Mean residual  : {val_df['residual'].mean():+.4f}  (+ = under, - = over)")
print(f"Std  residual  : {val_df['residual'].std():.4f}")
print('='*45)

## 9. Save Model

In [ ]:
save_dir = NOTEBOOKS_DIR / '08_forecasting'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(best_model, save_dir / 'xgb_safe_lag_v3_model.pkl')
print(f'Saved copy : {save_dir / "xgb_safe_lag_v3_model.pkl"}')
print(f'Main model : {MODELS_DIR / "xgboost_safe_lag_v3_model.pkl"}')
print(f'\nn_features_in_: {best_model.n_features_in_}  (expected 47)')
print(f'Feature names  : {list(X_train.columns)}')

joblib.dump(label_encoders, save_dir / 'xgb_v3_label_encoders.pkl')
print(f'Label encoders : {save_dir / "xgb_v3_label_encoders.pkl"}')